# Phase 1 — Legal Language Model (from scratch) · v2
**CS781 Capstone · Kaggle GPU notebook · Architecture #1 (decoder-only, MHA, sinusoidal)**

End-to-end pipeline for one Phase-1 experiment:

1. Stream the gated `Exploration-Lab/CS-781-Capstone` Phase-1 data (no 19 GB download).
2. Split **by document** into train / held-out validation, and build a **local dev set** (512-word prompt → 128-word reference) from the held-out documents.
3. Train **your A1 byte-level BPE** tokenizer, encode the corpus in parallel, save `train.bin` / `val.bin`, and reuse them in every architecture notebook (same data, so the comparison is controlled).
4. Build **your A3 DecoderLM** (50M–300M params), train it from scratch with resumable checkpoints.
5. Evaluate locally (val loss/perplexity + ROUGE-L / BLEU-4 / BERTScore on the dev set) and try different decoding settings.
6. Generate exactly 128 words for each official test prompt, check the output, and write `prediction.json` → `submission.zip`.
7. Log the run to `experiments.jsonl` and export the checkpoint + code for Phases 2–7.

---
### What changed from draft-1 (and why) — you must be able to explain each of these in the viva
| # | Problem seen in draft-1 output | Fix |
|---|---|---|
| FIX-1 | Smoke-test val loss was **15.2 → 10.4**, which is *higher* than `ln(2000)=7.6` (the loss of a uniform guess). Cause: no weight init. `nn.Embedding` defaults to N(0,1) and is **tied** to the output head, so initial logits have std ≈ √d_model ≈ 16, which makes the loss very large. That is why the output was "the the the …". | GPT-style init N(0, 0.02) for Linear/Embedding, plus a check that the initial loss is ≈ ln(V). |
| FIX-2 | Once embeddings are N(0, 0.02), the token signal is ~50× smaller than the sinusoidal position signal (amplitude ~1). | Multiply token embeddings by √d_model, as in the original Transformer. One marked line in `model_arch.py`, controlled by `scale_emb`. |
| FIX-3 | A3 default is **post-LN** (`norm_first=False`). Post-LN often trains unstably at 10+ layers without very careful warmup (Xiong et al. 2020). | `norm_first=True` (pre-LN). This flag already exists in your A3 code. |
| FIX-4 | Word counts were **78–80**, but Phase 1 needs 128 words. Generation used a fixed token cap and also stopped at `</s>`. | Generate until ≥129 words, then cut to 128. Special tokens are masked during sampling. |
| FIX-5 | Generating one prompt at a time took 12 min even for the tiny model. With the real model it would take hours. | Batched generation. Prompts in a batch are cut to the same token length, so no padding mask is needed. With **GPU T4 x2**, batches are split across **both GPUs** (about 2x faster; identical output). Progress prints at least every 5 min. |
| FIX-6 | `hf_hub_download` downloaded all ~19 GB of training JSON even in smoke mode. | Stream with `HfFileSystem`, so only the bytes you use are read. |
| FIX-7 | The config used 20M train tokens but consumed 4000×32×1024 ≈ 131M tokens, so the same data was seen **~6.5 times**. Validation was the token tail of the stream (train and val could share a document). | Split by document, **stratified per source** (equal val tokens and dev prompts from each of the 4 sources; val loss also reported per source). The data budget (`train_tokens`) is a **fixed** number (= what arch-1 consumes), independent of each architecture's steps/batch, so every architecture gets the *identical* split and tokenizer. A tokenizer fingerprint is checked on resume. |
| FIX-8 | Validation used 40 *random* batches, so it was noisy and not comparable across architectures. | Fixed, evenly spaced validation windows (the same for every run). |
| FIX-9 | Only the best weights were saved. A Kaggle timeout or disconnect lost the whole run. | Resumable `ckpt_last.pt` (model + optimizer + scaler + step + RNG), plus a **session clock**: training stops early enough that generation + zip always finish inside Kaggle's 12 h limit. Test predictions are written *before* the optional dev evaluation. |
| FIX-10 | No local evaluation (the document, Sec 16, asks for it). | Local dev set built from held-out documents, scored with ROUGE-L, BLEU-4 and BERTScore. |
| misc | Weight decay was also applied to LayerNorm and bias. LR decayed all the way to 0. The prompt could end mid-word. | Separate no-decay parameter group, `min_lr_ratio`, and a trailing space after the prompt. |

### How to run on Kaggle
1. Settings → Accelerator **GPU T4 x2** (training uses one GPU, generation uses both), Internet **On**. Add the Secret `HF_TOKEN`.
2. `SMOKE_TEST=True` first (~15 min). Check: initial loss ≈ ln(V), word counts = 128, zip passes the checks.
3. `SMOKE_TEST=False` → **Save Version → Save & Run All (Commit)**. Commit mode runs in the background for up to 12 h.
4. When it finishes, create a **Kaggle Dataset from the output**. Point `prebuilt_dir` at its `data_*` folder in the other architecture notebooks, and `resume_from` at `ckpt_last.pt` if a run was cut off.

In [ ]:
# 1 · Install dependencies (torch is already on Kaggle). Requires Internet = ON.
!pip -q install -U "huggingface_hub>=0.23" ijson rouge-score sacrebleu bert-score

In [ ]:
# 2 · Configuration ------------------------------------------------------------
RUN_NAME   = "arch1_mha_sinusoidal"      # names the run folder, checkpoints, logs and zip
SMOKE_TEST = True                        # True = tiny end-to-end check; False = real run

CONFIG = dict(
    hf_repo   = "Exploration-Lab/CS-781-Capstone",
    phase_dir = "Phase_1_Pretraining",
    sources   = ["acts", "high_courts", "supreme_court", "tribunals"],
    mix       = {"acts": 1, "high_courts": 1, "supreme_court": 1, "tribunals": 1},  # relative char-budget per source (a data-mixing experiment)
    prebuilt_dir = None,        # e.g. "/kaggle/input/<your-dataset>/data_bpe8000" -> reuse tokenizer + tokens (skip steps 5-6)

    # ---- tokenizer (your A1 byte-level BPE) ----
    vocab_size      = 8000,
    tok_train_chars = 12_000_000,

    # ---- data budget / validation ----
    train_tokens   = 131_073_025,  # FIXED data budget (= 4000 steps x 32 x 1024 + 1025, what arch-1 uses).
                                   # Do NOT change between architectures: it defines the train/val split.
    val_tokens     = 2_000_000,
    val_doc_frac   = 0.02,      # fraction of documents per source held out (never trained on)
    n_dev_examples = 100,       # local dev prompts built from held-out docs

    # ---- model (your A3 DecoderLM) ----
    d_model=768, n_layers=10, n_heads=12, n_kv_heads=12, d_ff=3072,
    max_seq_len=1024, dropout=0.1, pos_emb_type="sinusoidal",
    norm_first = True,          # [FIX-3] pre-LN
    scale_emb  = True,          # [FIX-2] sqrt(d_model) token-embedding scale

    # ---- training ----
    batch_size=8, grad_accum=4, lr=3e-4, min_lr_ratio=0.1, weight_decay=0.1,
    warmup_steps=200, max_steps=4000, grad_clip=1.0,
    eval_every=250, eval_batches=40, ckpt_every=250,
    session_limit_hours = 11.5, # Kaggle kills a session at 12 h (counted from the FIRST cell, incl. data prep)
    reserve_hours_after_training = 3.0,  # time kept for test generation (~2-2.5 h) + zip + dev eval
    resume_from = None,         # path to a previous ckpt_last.pt (e.g. /kaggle/input/.../ckpt_last.pt)

    # ---- generation ----
    gen_words=128, max_new_tokens=1024, gen_batch_size=16, gen_ctx_tokens=None,  # cap only; stops early at 128 words
    temperature=0.8, top_k=50, top_p=0.95, repetition_penalty=1.0,             # 1.0 = off
    gen_gpus = None,            # None = use every visible GPU for generation (Kaggle 'GPU T4 x2' -> 2x faster)
    run_bertscore = True,       # local BERTScore on the dev set (downloads roberta-large once)

    seed=1337, out_dir="/kaggle/working",
)

if SMOKE_TEST:
    CONFIG.update(
        vocab_size=2000, tok_train_chars=1_000_000, train_tokens=246_017, val_tokens=50_000, val_doc_frac=0.05, n_dev_examples=16,
        d_model=256, n_layers=4, n_heads=4, n_kv_heads=4, d_ff=1024, max_seq_len=256,
        batch_size=8, grad_accum=1, warmup_steps=20, max_steps=120,
        eval_every=40, eval_batches=10, ckpt_every=40, run_bertscore=False,
    )

import os, random, math, json, time, pickle, zipfile, shutil
SESSION_START = time.time()                      # the 12 h Kaggle clock starts (roughly) here
def session_hours(): return (time.time() - SESSION_START) / 3600
from dataclasses import asdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"

OUT_ROOT = CONFIG["out_dir"]
RUN_DIR  = os.path.join(OUT_ROOT, "runs", RUN_NAME + ("_smoke" if SMOKE_TEST else ""))
DATA_DIR = CONFIG["prebuilt_dir"] or os.path.join(
    OUT_ROOT, f"data_bpe{CONFIG['vocab_size']}" + ("_smoke" if SMOKE_TEST else ""))
os.makedirs(RUN_DIR, exist_ok=True)
TOKENS_PER_STEP = CONFIG["batch_size"] * CONFIG["grad_accum"] * CONFIG["max_seq_len"]
TRAIN_TOKENS = CONFIG["train_tokens"]
print("Device:", device, "| SMOKE_TEST:", SMOKE_TEST, "| run dir:", RUN_DIR, "| data dir:", DATA_DIR)
print(f"Tokens per optimizer step: {TOKENS_PER_STEP:,} | tokens the run will consume: {CONFIG['max_steps']*TOKENS_PER_STEP:,}")

In [ ]:
# 3 · Hugging Face login (the dataset is gated) --------------------------------
# Accept the dataset terms once at https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone
from huggingface_hub import login
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print("Loaded HF token from Kaggle Secrets.")
except Exception:
    pass
if not token:
    import getpass
    token = os.environ.get("HF_TOKEN") or getpass.getpass("Enter your Hugging Face token: ")
login(token=token)
print("Logged in to Hugging Face.")

## 4 · Your Assignment code (A3 model + A1 tokenizer)
`rope.py`, `attention.py` and `a1_tokenizer.py` are your code, unchanged. `model_arch.py` has one marked change (**[FIX-2]**: a `scale_emb` flag, off by default).

In [ ]:
import os
os.makedirs('a3src', exist_ok=True)

In [ ]:
%%writefile a3src/rope.py
# TODO 3.1: rotary positional embeddings from scratch (split-half convention).
import torch


def build_rope_cache(max_seq_len, d_head, base=10000.0):
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))   # (d_head/2,)
    positions = torch.arange(max_seq_len).float()
    angles = torch.outer(positions, inv_freq)                                  # (T, d_head/2)
    angles = torch.cat([angles, angles], dim=-1)                               # (T, d_head)
    return angles.cos(), angles.sin()


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

In [ ]:
%%writefile a3src/attention.py
# TODO 3.2: MHA / GQA / MQA attention with optional RoPE applied to q and k.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0,
                 use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        if attention_scale is None:
            self.attention_scale = 1.0 / math.sqrt(self.d_head)
        else:
            self.attention_scale = float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            q = apply_rope(q, self.rope_cos[:T], self.rope_sin[:T])
            k = apply_rope(k, self.rope_cos[:T], self.rope_sin[:T])

        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.attention_scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, self.n_heads * self.d_head)
        return self.resid_dropout(self.out_proj(out))

In [ ]:
%%writefile a3src/model_arch.py
# TODO 3.2: pass use_rope through the decoder; only build sinusoidal PE when requested.
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None
    norm_first: bool = False
    scale_emb: bool = False            # [FIX-2] multiply token embeddings by sqrt(d_model) (Vaswani et al. 2017, Sec 3.4)


def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm_first = cfg.norm_first
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.cfg.scale_emb:             # [FIX-2]
            x = x * math.sqrt(self.cfg.d_model)
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T].unsqueeze(0)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)

In [ ]:
%%writefile a3src/a1_tokenizer.py
"""Assignment-1 byte-level BPE tokenizer, retrained on the Assignment-3 corpus.

Byte-level BPE (A-1 Q5 spec): byte symbols so no <unk> is needed, reserved tokens
<pad> <unk> <s> </s> at ids 0..3, each merge takes the most frequent adjacent pair
(ties broken by the lexicographically smallest merged bytes), and merges never cross
a pre-token (word / whitespace) boundary so decode() reconstructs the text exactly.
"""
import re
from collections import Counter

RESERVED = ("<pad>", "<unk>", "<s>", "</s>")
BASE = len(RESERVED)
SPLIT = re.compile(r"\S+|\s+")


def merge_sequence(ids, pair, new_id):
    a, b = pair
    out = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return tuple(out)


class BPETokenizer:
    def __init__(self):
        self.merges = {}
        self.ranks = {}
        self.itos = {}
        self.vocab_size = 0
        self.cache = {}

    def train(self, text, vocab_size):
        self.itos = {i: b"" for i in range(BASE)}
        for byte in range(256):
            self.itos[BASE + byte] = bytes((byte,))

        counts = Counter(SPLIT.findall(text))
        corpus = Counter()
        for word, freq in counts.items():
            ids = tuple(BASE + byte for byte in word.encode("utf-8"))
            corpus[ids] += freq

        n_merges = max(0, vocab_size - BASE - 256)
        next_id = BASE + 256
        while len(self.merges) < n_merges:
            pair_freq = Counter()
            for ids, freq in corpus.items():
                for j in range(len(ids) - 1):
                    pair_freq[(ids[j], ids[j + 1])] += freq
            if not pair_freq:
                break
            top = max(pair_freq.values())
            if top < 2:
                break
            best_pairs = [p for p, f in pair_freq.items() if f == top]
            pair = min(best_pairs, key=lambda p: self.itos[p[0]] + self.itos[p[1]])

            self.itos[next_id] = self.itos[pair[0]] + self.itos[pair[1]]
            self.merges[pair] = next_id
            self.ranks[pair] = len(self.ranks)

            new_corpus = Counter()
            for ids, freq in corpus.items():
                new_corpus[merge_sequence(ids, pair, next_id)] += freq
            corpus = new_corpus
            next_id += 1

        self.vocab_size = BASE + 256 + len(self.merges)

    def encode_chunk(self, raw_bytes):
        if raw_bytes in self.cache:
            return self.cache[raw_bytes]
        ids = [BASE + byte for byte in raw_bytes]
        while len(ids) >= 2:
            candidates = {}
            for j in range(len(ids) - 1):
                pair = (ids[j], ids[j + 1])
                if pair in self.ranks:
                    candidates[pair] = self.ranks[pair]
            if not candidates:
                break
            pair = min(candidates, key=candidates.get)
            ids = list(merge_sequence(ids, pair, self.merges[pair]))
        self.cache[raw_bytes] = ids
        return ids

    def encode(self, text):
        ids = []
        for chunk in SPLIT.findall(text):
            ids.extend(self.encode_chunk(chunk.encode("utf-8")))
        return ids

    def decode(self, ids):
        data = b""
        for i in ids:
            data += self.itos.get(i, b"")
        return data.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self.vocab_size


def train_tokenizer(train_text, vocab_size):
    tokenizer = BPETokenizer()
    tokenizer.train(train_text, vocab_size)
    return tokenizer

In [ ]:
# 4b · Imports, tokenizer save/load, AMP helper ---------------------------------
import sys, importlib
sys.path.insert(0, os.path.abspath("a3src"))
import rope, attention, model_arch, a1_tokenizer
for _m in (rope, attention, model_arch, a1_tokenizer): importlib.reload(_m)
from a1_tokenizer import BPETokenizer, train_tokenizer
from model_arch import LMConfig, DecoderLM

def save_tokenizer(tok, path):
    cache = tok.cache; tok.cache = {}            # don't pickle the (large) encode cache
    try:
        with open(path, "wb") as f: pickle.dump(tok, f)
    finally:
        tok.cache = cache

def load_tokenizer(path):
    with open(path, "rb") as f: return pickle.load(f)

PAD, UNK, BOS, EOS = 0, 1, 2, 3                  # reserved ids in your A1 tokenizer
_use_cuda = (device == "cuda")
def amp_autocast():
    return torch.autocast(device_type=("cuda" if _use_cuda else "cpu"), enabled=_use_cuda)

DATA_READY = os.path.exists(os.path.join(DATA_DIR, "meta.json"))
print("Imported your A3 DecoderLM + A1 BPE tokenizer. Prebuilt data found:", DATA_READY)

## 5 · Data: stream → document-level split → dev set
Skipped automatically if `DATA_DIR` already has `meta.json` (for example when `prebuilt_dir` points to a saved Kaggle dataset).
**Look at the printed samples**: are they real legal text? Do the dev prompts look like the official test prompts?

In [ ]:
# 5a · Discover files, download only the (small) test file ---------------------
from huggingface_hub import HfApi, HfFileSystem, hf_hub_download
api = HfApi()
repo_files  = api.list_repo_files(CONFIG["hf_repo"], repo_type="dataset")
phase_files = [f for f in repo_files if f.startswith(CONFIG["phase_dir"] + "/")]
print("Phase-1 files:", *phase_files, sep="\n  ")

train_files = {}
for src in CONFIG["sources"]:
    m = [f for f in phase_files if "/Train/" in f and src in f.lower() and f.endswith(".json")]
    if m: train_files[src] = m[0]
test_files = [f for f in phase_files if "/Test/" in f and f.endswith(".json")]
assert train_files and test_files, "Phase-1 train/test files not found."
test_path = hf_hub_download(CONFIG["hf_repo"], test_files[0], repo_type="dataset")
print("Train sources:", list(train_files), "| test file:", test_files[0])

In [ ]:
# 5b · Stream documents (no full download) and build the splits ------------------
import ijson
print("ijson backend:", ijson.backend)           # yajl2_c (C) is much faster than 'python'
fs = HfFileSystem()

def hf_open(fname):
    return fs.open(f"datasets/{CONFIG['hf_repo']}/{fname}", "rb", block_size=16 * 1024 * 1024)

def iter_texts(obj):
    """Yield strings from a record of unknown shape (dict/list/str)."""
    if isinstance(obj, str):
        if obj.strip(): yield obj
    elif isinstance(obj, dict):
        for k in ("text", "content", "body", "document", "judgment", "raw_text"):
            if isinstance(obj.get(k), str) and obj[k].strip():
                yield obj[k]; return
        for v in obj.values():
            yield from iter_texts(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from iter_texts(v)

def toplevel_kind(fname):
    with hf_open(fname) as f:
        while True:
            c = f.read(1)
            if not c: return None
            if c.strip(): return c                # b"{" or b"["

def stream_docs(fname, max_chars):
    kind, got = toplevel_kind(fname), 0
    with hf_open(fname) as f:
        gen = ijson.kvitems(f, "") if kind == b"{" else ijson.items(f, "item")
        for item in gen:
            val = item[1] if kind == b"{" else item
            for t in iter_texts(val):
                yield t; got += len(t)
                if got >= max_chars: return

def collect_docs(fname, budget, tries=3):
    """Streaming ~200 MB over the network can hiccup; retry the whole source (same docs every time)."""
    for attempt in range(1, tries + 1):
        try:
            return list(stream_docs(fname, budget))
        except Exception as e:
            print(f"  stream error on {fname} ({type(e).__name__}: {e}); retry {attempt}/{tries}")
            time.sleep(15)
    raise RuntimeError(f"could not stream {fname}")

def build_dev_set(docs, n, prompt_words=512, cont_words=128, seed=0):
    """Mimic the official task: a 512-word prompt and the next 128 words as reference."""
    rng = random.Random(seed)
    need = prompt_words + cont_words
    pools = [d.split() for d in docs]
    pools = [w for w in pools if len(w) >= need] or [sum(pools, [])]   # fallback: concatenate
    prompts, refs = {}, {}
    for i in range(n):
        w = rng.choice(pools)
        s = rng.randint(0, len(w) - need)
        prompts[str(i)] = {"text": " ".join(w[s:s + prompt_words])}
        refs[str(i)]    = {"text": " ".join(w[s + prompt_words:s + need])}
    return prompts, refs

if not DATA_READY:
    CHARS_PER_TOKEN_GUESS = 6          # generous over-estimate; the true value is printed after encoding
    char_budget = max(TRAIN_TOKENS * CHARS_PER_TOKEN_GUESS, CONFIG["tok_train_chars"] * 1.2) / (1 - CONFIG["val_doc_frac"])
    wsum = sum(CONFIG["mix"][s] for s in train_files)
    rng = random.Random(CONFIG["seed"])
    train_pairs, val_by_src, source_stats = [], {}, {}
    for src, fname in train_files.items():
        budget = int(char_budget * CONFIG["mix"][src] / wsum)
        t0 = time.time()
        docs = collect_docs(fname, budget)
        rng.shuffle(docs)
        n_val = max(1, int(round(len(docs) * CONFIG["val_doc_frac"])))
        val_by_src[src] = docs[:n_val]                      # held out PER SOURCE (stratified)
        train_pairs += [(src, d) for d in docs[n_val:]]
        source_stats[src] = dict(docs=len(docs), chars=sum(map(len, docs)), val_docs=n_val)
        print(f"{src:>14}: {len(docs):>7,} docs | {source_stats[src]['chars']:>13,} chars | "
              f"{n_val} held out | {time.time()-t0:.0f}s")
    rng.shuffle(train_pairs)
    train_src = [s for s, _ in train_pairs]; train_docs = [d for _, d in train_pairs]; del train_pairs
    print(f"\nTrain docs: {len(train_docs):,} | Val docs: {sum(map(len, val_by_src.values())):,}")
    print("\n--- SAMPLE train doc ---\n", train_docs[0][:500])
    os.makedirs(DATA_DIR, exist_ok=True)
    # dev set: equal number of examples from every source's held-out docs
    dev_prompts, dev_refs, per = {}, {}, max(1, CONFIG["n_dev_examples"] // len(val_by_src))
    for j, (src, vdocs) in enumerate(val_by_src.items()):
        p, r = build_dev_set(vdocs, per, seed=CONFIG["seed"] + j)
        for k in p:
            key = str(len(dev_prompts))
            dev_prompts[key] = {**p[k], "source": src}; dev_refs[key] = {**r[k], "source": src}
    json.dump(dev_prompts, open(os.path.join(DATA_DIR, "dev_prompts.json"), "w"), ensure_ascii=False)
    json.dump(dev_refs,    open(os.path.join(DATA_DIR, "dev_refs.json"), "w"), ensure_ascii=False)
    print("\n--- SAMPLE dev prompt (last 200 chars) ---\n", dev_prompts["0"]["text"][-200:])
    print("--- its reference (first 200 chars) ---\n", dev_refs["0"]["text"][:200])

In [ ]:
# 6 · Train (or load) your byte-level BPE tokenizer -----------------------------
tok_path = os.path.join(DATA_DIR, "legal_bpe.pkl")
if os.path.exists(tok_path):
    tokenizer = load_tokenizer(tok_path)
    print("Loaded tokenizer from", tok_path)
else:
    parts, n = [], 0                               # train-split docs only (no val leakage)
    for d in train_docs:
        parts.append(d); n += len(d) + 2
        if n >= CONFIG["tok_train_chars"]: break
    tok_text = "\n\n".join(parts)[:CONFIG["tok_train_chars"]]
    print(f"Training BPE (vocab={CONFIG['vocab_size']}) on {len(tok_text):,} chars ... (one-time)")
    t0 = time.time()
    tokenizer = train_tokenizer(tok_text, CONFIG["vocab_size"])
    save_tokenizer(tokenizer, tok_path)
    print(f"Done in {time.time()-t0:.0f}s -> {tok_path}")

VOCAB = tokenizer.get_vocab_size()
import hashlib                                    # fingerprint: detects a mismatched tokenizer/checkpoint
TOK_HASH = hashlib.md5(json.dumps(sorted([a, b, c] for (a, b), c in tokenizer.merges.items())).encode()).hexdigest()[:12]
print("Tokenizer fingerprint:", TOK_HASH)
assert VOCAB < 65535, "token ids are stored as uint16"
s = "In the Supreme Court of India, the appellant contended that Section 302 IPC"
print("Vocab size:", VOCAB, "| round-trip OK:", tokenizer.decode(tokenizer.encode(s)) == s)
print("Pieces:", [tokenizer.decode([i]) for i in tokenizer.encode(s)])

In [ ]:
# 6b · Encode the corpus in parallel -> train.bin / val.bin (uint16) -----------
from multiprocessing import Pool

def _encode_doc(doc):
    ids = tokenizer.encode(doc); ids.append(EOS)
    return np.asarray(ids, dtype=np.uint16)

def encode_docs(docs, max_tokens, workers=os.cpu_count() or 2):
    """Returns (token array, chars consumed, per-doc token counts of the docs used)."""
    out, n, chars, lens, t0 = np.empty(max_tokens, dtype=np.uint16), 0, 0, [], time.time()
    with Pool(workers) as pool:
        for i, arr in enumerate(pool.imap(_encode_doc, docs, chunksize=8)):
            k = min(len(arr), max_tokens - n)
            out[n:n + k] = arr[:k]; n += k; chars += len(docs[i]); lens.append(k)
            if i % 2000 == 0:
                print(f"  {i:,} docs -> {n:,} tokens ({time.time()-t0:.0f}s)")
            if n >= max_tokens: break
    return out[:n], chars, lens

if not DATA_READY:
    # validation: equal token budget per source, stored as contiguous blocks (ranges saved in meta)
    val_parts, val_ranges, pos = [], {}, 0
    for src, vdocs in val_by_src.items():
        arr, _, _ = encode_docs(vdocs, CONFIG["val_tokens"] // len(val_by_src))
        val_parts.append(arr); val_ranges[src] = [pos, pos + len(arr)]; pos += len(arr)
    val_arr = np.concatenate(val_parts)
    train_arr, train_chars, lens = encode_docs(train_docs, TRAIN_TOKENS)
    train_tokens_by_source = {s: 0 for s in val_by_src}
    for s, k in zip(train_src, lens): train_tokens_by_source[s] += k
    train_arr.tofile(os.path.join(DATA_DIR, "train.bin"))
    val_arr.tofile(os.path.join(DATA_DIR, "val.bin"))
    meta = dict(vocab_size=VOCAB, n_train=int(len(train_arr)), n_val=int(len(val_arr)),
                chars_per_token=round(train_chars / max(1, len(train_arr)), 3),
                train_tokens_by_source=train_tokens_by_source, val_ranges=val_ranges,
                mix=CONFIG["mix"], source_stats=source_stats, val_doc_frac=CONFIG["val_doc_frac"],
                tok_train_chars=CONFIG["tok_train_chars"], seed=CONFIG["seed"], tok_hash=TOK_HASH,
                train_tokens_budget=TRAIN_TOKENS)
    json.dump(meta, open(os.path.join(DATA_DIR, "meta.json"), "w"), indent=2)
    del train_docs, train_src, train_arr, val_arr, val_parts, val_by_src   # free RAM
    DATA_READY = True

meta = json.load(open(os.path.join(DATA_DIR, "meta.json")))
train_data = np.memmap(os.path.join(DATA_DIR, "train.bin"), dtype=np.uint16, mode="r")
val_data   = np.memmap(os.path.join(DATA_DIR, "val.bin"),   dtype=np.uint16, mode="r")
dev_prompts = json.load(open(os.path.join(DATA_DIR, "dev_prompts.json"), encoding="utf-8"))
dev_refs    = json.load(open(os.path.join(DATA_DIR, "dev_refs.json"),    encoding="utf-8"))
epochs = CONFIG["max_steps"] * TOKENS_PER_STEP / len(train_data)
print(f"Train tokens: {len(train_data):,} | Val tokens: {len(val_data):,} | "
      f"chars/token: {meta['chars_per_token']} | dev examples: {len(dev_prompts)}")
print(f"This run will make {epochs:.2f} passes over the training tokens.")
if epochs > 1.05: print("NOTE: >1 pass means repeated data (this architecture uses more steps/tokens than arch-1).")
assert meta.get("tok_hash") in (None, TOK_HASH), "tokenizer does not match the one this data was encoded with"
for src, n in meta.get("train_tokens_by_source", {}).items():
    r = meta.get("val_ranges", {}).get(src, [0, 0])
    print(f"  {src:>14}: train {n:>12,} tokens ({n / max(1, len(train_data)):.1%}) | val {r[1]-r[0]:>10,} tokens")

## 7 · Model: build, initialise, sanity-check
The **initial loss must be ≈ ln(V)**. A randomly initialised model should be about as good as a uniform guess. Draft-1 started far above ln(V) (see FIX-1).

In [ ]:
# 7 · Build your DecoderLM, init weights, check the parameter budget -----------
cfg = LMConfig(
    vocab_size=VOCAB, d_model=CONFIG["d_model"], n_layers=CONFIG["n_layers"],
    n_heads=CONFIG["n_heads"], n_kv_heads=CONFIG["n_kv_heads"], d_ff=CONFIG["d_ff"],
    max_seq_len=CONFIG["max_seq_len"], dropout=CONFIG["dropout"], pos_emb_type=CONFIG["pos_emb_type"],
    norm_first=CONFIG["norm_first"], scale_emb=CONFIG["scale_emb"],
)
model = DecoderLM(cfg)

def init_weights(m):                              # [FIX-1] GPT-2 style initialisation
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, mean=0.0, std=0.02)
model.apply(init_weights)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())      # tied weights are counted once
n_emb = model.tok_emb.weight.numel()
print(f"Parameters: {n_params:,} (non-embedding {n_params - n_emb:,})")
if not SMOKE_TEST:
    assert 50_000_000 <= n_params <= 300_000_000, "Phase-1 model must be 50M-300M params."
    print("Within the 50M-300M Phase-1 budget.")

T, BS, ACC = cfg.max_seq_len, CONFIG["batch_size"], CONFIG["grad_accum"]
assert len(val_data) > T + 2 and len(train_data) > T + 2, "not enough tokens for one window"
g_batch = torch.Generator().manual_seed(CONFIG["seed"])

def get_batch(data, bs, starts=None):
    if starts is None:
        starts = torch.randint(0, len(data) - T - 1, (bs,), generator=g_batch).tolist()
    x = torch.from_numpy(np.stack([data[s:s + T] for s in starts]).astype(np.int64))
    y = torch.from_numpy(np.stack([data[s + 1:s + 1 + T] for s in starts]).astype(np.int64))
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)

# [FIX-8] fixed, evenly spaced validation windows -> comparable across runs/architectures.
# Equal number of windows inside each source's block of val.bin (sources are weighted equally).
n_win = CONFIG["eval_batches"] * BS
ranges = meta.get("val_ranges") or {"all": [0, len(val_data)]}
ok = {s: r for s, r in ranges.items() if r[1] - r[0] > T + 2} or {"all": [0, len(val_data)]}
if len(ok) < len(ranges): print("WARNING: some sources have too few val tokens:", set(ranges) - set(ok))
VAL_STARTS_BY_SRC = {s: np.linspace(a, b - T - 2, max(1, n_win // len(ok))).astype(int).tolist()
                     for s, (a, b) in ok.items()}
VAL_STARTS = sum(VAL_STARTS_BY_SRC.values(), [])

@torch.no_grad()
def evaluate(starts=None):
    starts = starts or VAL_STARTS
    was_training = model.training; model.eval(); total = 0.0
    for i in range(0, len(starts), BS):
        x, y = get_batch(val_data, BS, starts[i:i + BS])
        with amp_autocast():
            logits = model(x)
        total += F.cross_entropy(logits.float().reshape(-1, VOCAB), y.reshape(-1)).item() * x.size(0)
    model.train(was_training)
    l = total / len(starts)
    return l, math.exp(min(20, l))

def evaluate_by_source():
    return {s: dict(zip(("val_loss", "val_ppl"), evaluate(st))) for s, st in VAL_STARTS_BY_SRC.items()}

init_loss, _ = evaluate()
print(f"Initial val loss {init_loss:.3f} vs ln(V) = {math.log(VOCAB):.3f}")
if init_loss > math.log(VOCAB) + 1.0:
    print("WARNING: initial loss is far above ln(V) -> check initialisation.")

In [ ]:
# 8 · Train from scratch (resumable) --------------------------------------------
try:
    scaler = torch.amp.GradScaler("cuda", enabled=_use_cuda)
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=_use_cuda)

def lr_at(step):
    w, total, peak = CONFIG["warmup_steps"], CONFIG["max_steps"], CONFIG["lr"]
    if step < w: return peak * step / max(1, w)
    p = min(1.0, (step - w) / max(1, total - w))
    lo = peak * CONFIG["min_lr_ratio"]
    return lo + 0.5 * (peak - lo) * (1 + math.cos(math.pi * p))

# no weight decay on biases / LayerNorm (1-D params)
decay    = [p for _, p in model.named_parameters() if p.dim() >= 2]
no_decay = [p for _, p in model.named_parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": CONFIG["weight_decay"]},
                         {"params": no_decay, "weight_decay": 0.0}],
                        lr=CONFIG["lr"], betas=(0.9, 0.95))

CK_LAST = os.path.join(RUN_DIR, "ckpt_last.pt")
CK_BEST = os.path.join(RUN_DIR, "ckpt_best.pt")
HIST    = os.path.join(RUN_DIR, "history.json")

def atomic_save(obj, path):
    torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path)

def save_last(step, best, history):
    atomic_save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                 "step": step, "best": best, "history": history, "cfg": asdict(cfg), "config": CONFIG, "tok_hash": TOK_HASH,
                 "torch_rng": torch.get_rng_state(), "g_batch": g_batch.get_state(),
                 "cuda_rng": torch.cuda.get_rng_state_all() if _use_cuda else None}, CK_LAST)

def save_best(step, vl):
    atomic_save({"model": model.state_dict(), "cfg": asdict(cfg), "config": CONFIG, "tok_hash": TOK_HASH,
                 "step": step, "val_loss": vl}, CK_BEST)

# Resume: a ckpt_last.pt already in this run folder (same session) is always the newest;
# otherwise use CONFIG["resume_from"] (e.g. a checkpoint attached from /kaggle/input).
start_step, best, history = 1, float("inf"), []
resume = CK_LAST if os.path.exists(CK_LAST) else CONFIG["resume_from"]
if resume:
    assert os.path.exists(resume), f"resume_from not found: {resume}"
    st = torch.load(resume, map_location=device, weights_only=False)
    assert st.get("tok_hash") in (None, TOK_HASH), (
        f"Checkpoint was trained with a different tokenizer ({st['tok_hash']} vs {TOK_HASH}): wrong data attached?")
    model.load_state_dict(st["model"])
    if "opt" not in st:
        # a ckpt_best.pt (weights only): no training, go straight to evaluation + generation
        if not os.path.exists(CK_BEST): shutil.copy(resume, CK_BEST)
        start_step, best = CONFIG["max_steps"] + 1, st.get("val_loss", float("inf"))
        print(f"Loaded trained weights from {resume} (step {st.get('step')}) -> skipping training.")
    else:
        opt.load_state_dict(st["opt"]); scaler.load_state_dict(st["scaler"])
        torch.set_rng_state(st["torch_rng"]); g_batch.set_state(st["g_batch"])
        if _use_cuda and st.get("cuda_rng") is not None: torch.cuda.set_rng_state_all(st["cuda_rng"])
        start_step, best, history = st["step"] + 1, st["best"], st["history"]
        src_best = os.path.join(os.path.dirname(os.path.abspath(resume)), "ckpt_best.pt")
        if not os.path.exists(CK_BEST) and os.path.exists(src_best):
            shutil.copy(src_best, CK_BEST)                  # bring the matching best checkpoint along
        if not os.path.exists(CK_BEST): best = float("inf")
        print(f"Resumed from {resume} at step {start_step} (best val so far {best:.3f})")

train_deadline_h = CONFIG["session_limit_hours"] - CONFIG["reserve_hours_after_training"]
print(f"Session time used so far: {session_hours():.2f} h | training will stop at {train_deadline_h:.1f} h")

LOG_EVERY = max(1, CONFIG["eval_every"] // 5)
model.train(); t0 = t_log = time.time(); run_loss = torch.zeros((), device=device); n_run = 0
step = start_step - 1
for step in range(start_step, CONFIG["max_steps"] + 1):
    lr = lr_at(step)
    for g in opt.param_groups: g["lr"] = lr
    opt.zero_grad(set_to_none=True)
    for _ in range(ACC):
        x, y = get_batch(train_data, BS)
        with amp_autocast():
            loss = F.cross_entropy(model(x).float().reshape(-1, VOCAB), y.reshape(-1)) / ACC
        scaler.scale(loss).backward()
        run_loss += loss.detach()
    scaler.unscale_(opt)
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
    scaler.step(opt); scaler.update(); n_run += 1

    if step % LOG_EVERY == 0:
        dt = time.time() - t_log; t_log = time.time()
        tl = (run_loss / n_run).item(); run_loss.zero_(); n_run = 0
        eta = (CONFIG["max_steps"] - step) * dt / LOG_EVERY / 60
        print(f"step {step:>5} | lr {lr:.2e} | train {tl:.3f} | gnorm {float(gnorm):.2f} | "
              f"{LOG_EVERY * TOKENS_PER_STEP / dt:,.0f} tok/s | ETA {eta:.0f} min")
    if step % CONFIG["eval_every"] == 0 or step == CONFIG["max_steps"]:
        vl, vppl = evaluate()
        history.append(dict(step=step, tokens=step * TOKENS_PER_STEP, lr=lr, val_loss=vl, val_ppl=vppl,
                            train_loss=tl if step % LOG_EVERY == 0 else None,
                            elapsed_min=(time.time() - t0) / 60))
        json.dump(history, open(HIST, "w"), indent=1)
        msg = f"  >> eval step {step}: val loss {vl:.3f} | val ppl {vppl:.1f}"
        if vl < best:
            best = vl; save_best(step, vl)
            msg += "  (new best, saved)"
        print(msg)
    if step % CONFIG["ckpt_every"] == 0:
        save_last(step, best, history)
    if session_hours() > train_deadline_h:
        print(f"Session time {session_hours():.1f} h reached at step {step}: stopping training so that "
              f"generation + zip still finish before Kaggle's 12 h limit (resume later with resume_from).")
        break
if step >= start_step: save_last(step, best, history)   # only if this session actually trained
if not os.path.exists(CK_BEST):                     # e.g. resumed at the final step with no best ckpt
    vl, _ = evaluate(); best = vl; save_best(step, vl)
TRAIN_MINUTES = (time.time() - t0) / 60
print(f"Training done. Best val loss {best:.3f} (ppl {math.exp(min(20, best)):.1f}) in {TRAIN_MINUTES:.0f} min.")

In [ ]:
# 8b · Loss curves (saved for the report) ----------------------------------------
import matplotlib.pyplot as plt
h = [r for r in history if r["val_loss"] is not None]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([r["step"] for r in h], [r["val_loss"] for r in h], "o-", label="val")
tr = [r for r in h if r["train_loss"] is not None]
ax[0].plot([r["step"] for r in tr], [r["train_loss"] for r in tr], "s--", label="train (recent avg)")
ax[0].set_xlabel("step"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title(RUN_NAME)
ax[1].plot([r["tokens"] / 1e6 for r in h], [r["val_ppl"] for r in h], "o-")
ax[1].set_xlabel("tokens seen (M)"); ax[1].set_ylabel("val perplexity"); ax[1].set_yscale("log")
plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "loss_curves.png"), dpi=120); plt.show()

In [ ]:
# 8c · Load the best checkpoint for generation ------------------------------------
state = torch.load(CK_BEST, map_location=device, weights_only=False)
model.load_state_dict(state["model"]); model.eval()
print(f"Loaded best checkpoint (step {state['step']}, val loss {state['val_loss']:.3f}).")

In [ ]:
# 9 · Batched generation: temperature / top-k / top-p / repetition penalty -------
SPECIAL_IDS = [PAD, UNK, BOS, EOS]

def first_n_words(text, n):
    return " ".join(text.split()[:n])

@torch.no_grad()
def generate_batch(enc_prompts, gen_words, max_new_tokens, temperature, top_k, top_p,
                   repetition_penalty=1.0, ctx_tokens=None, seed=0, net=None, dev=None):
    """enc_prompts: list of token-id lists. All prompts are cut to the same length P (their last P
    tokens), so the batch needs no padding mask. Generates until every row has > gen_words words.
    net/dev: which model copy / device to use (for multi-GPU generation); default = model/device."""
    net = net or model; dev = dev or next(net.parameters()).device
    Tm = cfg.max_seq_len
    P = min(min(len(e) for e in enc_prompts), ctx_tokens or Tm - 1)
    ctx = torch.tensor([e[-P:] for e in enc_prompts], dtype=torch.long, device=dev)
    gen = torch.Generator(device=dev); gen.manual_seed(seed)
    for step in range(max_new_tokens):
        window = ctx[:, -Tm:]                                  # sliding window beyond max_seq_len
        with amp_autocast():
            logits = net(window)[:, -1, :].float()
        logits[:, SPECIAL_IDS] = float("-inf")                 # [FIX-4] never stop / emit specials
        if repetition_penalty != 1.0:                          # CTRL-style penalty on tokens in the window
            sc = logits.gather(1, window)
            sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
            logits.scatter_(1, window, sc)
        if temperature <= 0:                                   # greedy
            nxt = logits.argmax(-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            if top_p and top_p < 1.0:
                sp, si = torch.sort(probs, descending=True)
                keep = (torch.cumsum(sp, dim=-1) - sp) <= top_p
                sp = sp * keep; sp = sp / sp.sum(dim=-1, keepdim=True)
                nxt = si.gather(-1, torch.multinomial(sp, 1, generator=gen))
            else:
                nxt = torch.multinomial(probs, 1, generator=gen)
        ctx = torch.cat([ctx, nxt], dim=1)
        if (step + 1) % 16 == 0:                               # [FIX-4] stop once all rows have >128 words
            texts = [tokenizer.decode(r) for r in ctx[:, P:].tolist()]
            if all(len(t.split()) > gen_words for t in texts): break
    return [first_n_words(tokenizer.decode(r), gen_words) for r in ctx[:, P:].tolist()]

import copy, threading, queue

def gen_devices():
    """All GPUs to generate on (Kaggle 'GPU T4 x2' -> 2). CONFIG['gen_gpus'] can limit the number."""
    if not _use_cuda: return [torch.device(device)]
    n = torch.cuda.device_count()
    n = min(n, CONFIG["gen_gpus"]) if CONFIG.get("gen_gpus") else n
    return [torch.device(f"cuda:{i}") for i in range(max(1, n))]

def generate_all(prompt_dict, dec, bs=None, log_every=10, log_secs=300):
    """prompt_dict: {id: {"text": prompt}} -> {id: {"text": 128-word continuation}}.
    Batches are shared out across all GPUs through a work queue: one model copy + one thread per GPU
    (PyTorch releases the GIL inside CUDA ops, so the GPUs really run in parallel).
    Each batch keeps its own seed, so the output is the same whatever the number of GPUs."""
    bs = bs or CONFIG["gen_batch_size"]
    ids = list(prompt_dict)
    # trailing space: the continuation must start a NEW word, not finish the prompt's last word
    enc = {k: tokenizer.encode(prompt_dict[k]["text"].rstrip() + " ") for k in ids}
    order = sorted(ids, key=lambda k: len(enc[k]))             # similar lengths per batch
    starts = list(range(0, len(order), bs))
    devs = gen_devices()[:max(1, len(starts))]
    main_dev = next(model.parameters()).device
    replicas = [(model if d == main_dev else copy.deepcopy(model).to(d).eval(), d) for d in devs]
    work = queue.Queue()
    for b in starts: work.put(b)
    out, errors, lock = {}, [], threading.Lock()
    t0 = t_print = time.time(); done = [0]

    def worker(net, dev):
        try:
            while True:
                try: b = work.get_nowait()
                except queue.Empty: return
                chunk = order[b:b + bs]
                texts = generate_batch([enc[k] for k in chunk], CONFIG["gen_words"], CONFIG["max_new_tokens"],
                                       dec["temperature"], dec["top_k"], dec["top_p"],
                                       dec.get("repetition_penalty", 1.0), CONFIG["gen_ctx_tokens"],
                                       seed=CONFIG["seed"] + b, net=net, dev=dev)
                with lock:
                    for k, t in zip(chunk, texts): out[k] = {"text": t}
                    done[0] += 1
        except Exception as e:                                  # surface thread errors in the main thread
            errors.append(e)

    threads = [threading.Thread(target=worker, args=r, daemon=True) for r in replicas]
    for th in threads: th.start()
    if len(starts) > 1:
        print(f"  generating {len(order)} prompts in {len(starts)} batches on {len(devs)} device(s): "
              + ", ".join(map(str, devs)))
    last = -1
    while any(th.is_alive() for th in threads):
        time.sleep(2)
        with lock: n = done[0]
        if n != last and (n % log_every == 0 or time.time() - t_print > log_secs):
            if n > 0:
                eta = (time.time() - t0) / n * (len(starts) - n) / 60
                print(f"  {min(n * bs, len(order))}/{len(order)} done ({time.time()-t0:.0f}s, ETA {eta:.0f} min)")
            t_print, last = time.time(), n
    for th in threads: th.join()
    if errors: raise errors[0]
    del replicas
    if _use_cuda: torch.cuda.empty_cache()
    return {k: out[k] for k in ids}

DECODE = dict(temperature=CONFIG["temperature"], top_k=CONFIG["top_k"], top_p=CONFIG["top_p"],
              repetition_penalty=CONFIG["repetition_penalty"])
demo = generate_all({"d": {"text": "In the Supreme Court of India, the appellant contended that"}}, DECODE)
print(demo["d"]["text"])

In [ ]:
# 10 · Generate a continuation for every official test prompt ----------------------
with open(test_path, encoding="utf-8") as f:
    test_data = json.load(f)
assert isinstance(test_data, dict), f"Expected an id->record dict, got {type(test_data)}"

def get_text_field(v):
    if isinstance(v, str): return v
    if isinstance(v, dict):
        for k in ("text", "content", "prompt", "body"):
            if isinstance(v.get(k), str): return v[k]
        for vv in v.values():
            if isinstance(vv, str): return vv
    return ""

test_prompts = {k: {"text": get_text_field(v)} for k, v in test_data.items()}
wl = [len(p["text"].split()) for p in test_prompts.values()]
print(f"{len(test_prompts)} test prompts | words min {min(wl)} mean {np.mean(wl):.0f} max {max(wl)}")
print("Prompt tail:", repr(next(iter(test_prompts.values()))["text"][-150:]))
print("Decoding:", DECODE)
t0 = time.time()
preds = generate_all(test_prompts, DECODE)
print(f"Generated {len(preds)} continuations in {(time.time()-t0)/60:.1f} min.")

In [ ]:
# 11 · Validate, write prediction.json, zip, re-check the zip -----------------------
assert set(preds) == set(test_data), "Test IDs must match exactly (no missing/extra)."
for k, v in preds.items():
    assert isinstance(v["text"], str) and v["text"].strip(), f"Empty continuation for id {k}"
wc = [len(v["text"].split()) for v in preds.values()]
print(f"Word counts: min {min(wc)}, max {max(wc)}, mean {np.mean(wc):.1f} (target {CONFIG['gen_words']})")
if min(wc) < CONFIG["gen_words"]:
    print(f"WARNING: {sum(w < CONFIG['gen_words'] for w in wc)} outputs are short -> raise max_new_tokens.")

pred_path = os.path.join(RUN_DIR, "prediction.json")
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False)
zip_path = os.path.join(RUN_DIR, f"submission_{RUN_NAME}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")               # MUST be at the zip root
with zipfile.ZipFile(zip_path) as z:
    assert z.namelist() == ["prediction.json"], z.namelist()
    reread = json.loads(z.read("prediction.json"))
    assert set(reread) == set(test_data) and all("text" in v for v in reread.values())
print("Zip verified ->", zip_path, " (upload this file to Codabench)")

## 12 · Local evaluation on the held-out dev set (runs after the submission is safely written)
Val loss/perplexity measure the LM. The leaderboard scores **generation** with BERTScore, ROUGE-L and BLEU-4. Record both, because the brief asks you to study how they relate.
Add your own decoding settings to `DECODING_TRIALS`. Choosing among them, and explaining why, is your analysis. Then put the chosen values in `CONFIG`.
*Note: this local BERTScore uses `bert-score`'s default English model. Use the official evaluation scripts (Drive link in Sec 18.1) to match Codabench exactly, and compare runs relative to each other.*

In [ ]:
# 12 · Score continuations against references -------------------------------------
def score_predictions(preds, refs, bertscore=CONFIG["run_bertscore"]):
    ids = list(refs)
    hyps = [preds[k]["text"] for k in ids]; gold = [refs[k]["text"] for k in ids]
    res = {}
    try:
        from rouge_score import rouge_scorer
        rs = rouge_scorer.RougeScorer(["rougeL"])
        res["rougeL_f1"] = float(np.mean([rs.score(g, h)["rougeL"].fmeasure for g, h in zip(gold, hyps)]))
    except Exception as e: print("ROUGE skipped:", e)
    try:
        import sacrebleu
        res["bleu4"] = sacrebleu.corpus_bleu(hyps, [gold]).score / 100
    except Exception as e: print("BLEU skipped:", e)
    if bertscore:
        try:
            from bert_score import score as bscore
            _, _, F1 = bscore(hyps, gold, lang="en", batch_size=16, verbose=False)
            res["bertscore_f1"] = float(F1.mean())
            if _use_cuda: torch.cuda.empty_cache()
        except Exception as e: print("BERTScore skipped:", e)
    return res

DECODING_TRIALS = [
    dict(name="config", **DECODE),
    # add your own decoding settings here, one dict per trial
]
if session_hours() > CONFIG["session_limit_hours"] - 0.4:
    print(f"Only {12 - session_hours():.1f} h left in the session -> skipping decoding trials (val loss only).")
    DECODING_TRIALS = []
val_loss, val_ppl = evaluate()
dev_results = {"val_loss": val_loss, "val_ppl": val_ppl, "val_by_source": evaluate_by_source(), "decoding": []}
print(f"Val loss {val_loss:.3f} (ppl {val_ppl:.1f})")
for s, r in dev_results["val_by_source"].items():
    print(f"  {s:>14}: val loss {r['val_loss']:.3f} | ppl {r['val_ppl']:.1f}")
dev_sources = sorted({v.get("source", "all") for v in dev_refs.values()})
for trial in DECODING_TRIALS:
    t0 = time.time()
    dev_preds = generate_all(dev_prompts, trial, log_every=10**9)
    scores = score_predictions(dev_preds, dev_refs)
    if len(dev_sources) > 1:                     # same metrics per source (BERTScore off here: slow)
        scores["by_source"] = {s: score_predictions(dev_preds, {k: v for k, v in dev_refs.items()
                                                                 if v.get("source") == s}, bertscore=False)
                               for s in dev_sources}
    dev_results["decoding"].append({**trial, **scores, "gen_sec": round(time.time() - t0, 1)})
    print(trial["name"], {k: (round(v, 4) if isinstance(v, float) else v) for k, v in scores.items()})
json.dump(dev_results, open(os.path.join(RUN_DIR, "dev_results.json"), "w"), indent=1)
if DECODING_TRIALS:
    k0 = next(iter(dev_refs))
    print("\nREF :", dev_refs[k0]["text"][:300]); print("PRED:", dev_preds[k0]["text"][:300])

In [ ]:
# 13 · Log the experiment + export everything Phases 2-7 will need ------------------
record = dict(run=RUN_NAME, smoke=SMOKE_TEST, time=time.strftime("%Y-%m-%d %H:%M"),
              params=n_params, non_embedding_params=n_params - n_emb,
              best_step=state["step"], best_val_loss=state["val_loss"],
              best_val_ppl=math.exp(min(20, state["val_loss"])), train_minutes=round(TRAIN_MINUTES, 1),
              dev=dev_results, decoding_used=DECODE, data=meta, config=CONFIG,
              test_word_counts=[int(min(wc)), int(max(wc))])
json.dump(record, open(os.path.join(RUN_DIR, "record.json"), "w"), indent=1, default=str)
with open(os.path.join(OUT_ROOT, "experiments.jsonl"), "a") as f:
    f.write(json.dumps(record, default=str) + "\n")

shutil.copytree("a3src", os.path.join(RUN_DIR, "code"), dirs_exist_ok=True)   # exact code used
shutil.copy(os.path.join(DATA_DIR, "legal_bpe.pkl"), RUN_DIR)
print("Run folder contents:")
for fn in sorted(os.listdir(RUN_DIR)):
    p = os.path.join(RUN_DIR, fn)
    print(f"  {fn:<40} {os.path.getsize(p)/1e6:8.1f} MB" if os.path.isfile(p) else f"  {fn}/")

## Next steps
1. **Smoke run.** Check that the initial loss is ≈ ln(V), the val loss drops *below* ln(V), the word counts are exactly 128, and the zip verifies. Upload the smoke zip once only to test the Codabench flow (starter/smoke runs do not count toward the minimum).
2. **Real run of this architecture.** Set `SMOKE_TEST=False` and commit. Save the output as a Kaggle Dataset (it holds `data_bpe8000/` + `runs/<RUN_NAME>/`). This is **submission #1**.
3. **Architectures #2–#4 (your own design and code).** Copy this notebook, set `prebuilt_dir` to the saved `data_bpe8000`, add your model file under `a3src/`, and build it in cell 7. The rest of the pipeline only needs `model(idx) -> logits (B, T, V)`. Keep the data, validation windows and dev set the same so the comparison is fair. Use a separate Kaggle account or session per architecture to run them in parallel.
4. **Think ahead to Phase 2** (starts 7 Oct). It requires **all four** Phase-1 models, a token/span head (so each model should be able to return hidden states), and a knowledge-distillation experiment (larger teacher → smaller student). Keep every `ckpt_best.pt` + `code/` + `legal_bpe.pkl`.
5. **Report evidence.** `experiments.jsonl`, `history.json`, `loss_curves.png` and `dev_results.json` per run. The analysis (what worked, what failed, and why) has to be your own writing.